# RAGAS Metrics — Context + Correctness (gpt-4o-mini)

Calcul des métriques contexte et gold_answer avec **gpt-4o-mini** comme juge :
- `context_precision`, `context_recall` (nécessitent gold_answer)
- `answer_correctness`, `answer_similarity` (nécessitent gold_answer)

Résultats stockés sans suffixe dans le JSONB `metrics`.

**Lancer en parallèle** avec `eval_v3clean_metrics_faithfulness.ipynb` (gpt-4.1-mini).

In [ ]:
import os, sys, json, time
from pathlib import Path
from datetime import datetime

import pandas as pd
import psycopg
from psycopg.rows import dict_row
from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
load_dotenv(PROJECT_ROOT / '.env')

DSN = os.getenv('TUNNEL_DSN') or os.getenv('SCALINGO_POSTGRESQL_URL') or os.getenv('PG_DSN')
with psycopg.connect(DSN, row_factory=dict_row) as conn:
    cnt = conn.execute('SELECT count(*) as cnt FROM goldset_runs').fetchone()['cnt']
print(f'DB OK — {cnt} total runs')

In [ ]:
JUDGE_MODEL = 'gpt-4o-mini'
SUFFIX = ''  # pas de suffixe pour gpt-4o-mini
BATCH_SIZE = 50

CONFIGS_TO_PROCESS = [
    'v3clean_prod_wide',
]

print(f'Judge: {JUDGE_MODEL}')
print(f'Configs: {CONFIGS_TO_PROCESS}')

In [ ]:
from ragas import evaluate as ragas_evaluate
from ragas.metrics import context_precision, context_recall, answer_correctness, answer_similarity
from datasets import Dataset
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

ragas_llm = ChatOpenAI(model=JUDGE_MODEL, temperature=0)
ragas_embeddings = OpenAIEmbeddings()

METRICS = [context_precision, context_recall, answer_correctness, answer_similarity]
METRIC_KEYS = ['context_precision', 'context_recall', 'answer_correctness', 'answer_similarity']

test = ragas_llm.invoke('Dis OK.')
print(f'RAGAS ready: {JUDGE_MODEL} — {test.content}')

In [ ]:
_db_conn = None

def _get_conn(row_factory=None):
    global _db_conn
    if _db_conn is not None:
        try:
            _db_conn.execute('SELECT 1')
            if row_factory:
                _db_conn.row_factory = row_factory
            return _db_conn
        except Exception:
            try: _db_conn.close()
            except Exception: pass
            _db_conn = None
    _db_conn = psycopg.connect(DSN, **(dict(row_factory=row_factory) if row_factory else {}))
    return _db_conn


def load_runs(config_name: str) -> pd.DataFrame:
    """Load runs needing context/correctness metrics. Requires gold_answer."""
    sql = '''
        SELECT gr.id as run_id, gr.question_id, gq.question, gq.gold_answer,
               gq.goldset_name, gr.response, gr.retrieved_context, gr.metrics
        FROM goldset_runs gr
        JOIN goldset_questions_v2 gq ON gr.question_id = gq.id
        WHERE gr.config_name = %s
          AND gr.response IS NOT NULL
          AND gr.retrieved_context IS NOT NULL
          AND gq.gold_answer IS NOT NULL AND gq.gold_answer != ''
          AND (gr.metrics IS NULL
               OR NOT (gr.metrics ? 'context_precision')
               OR NOT (gr.metrics ? 'answer_correctness'))
        ORDER BY gr.id
    '''
    conn = _get_conn(row_factory=dict_row)
    return pd.DataFrame(conn.execute(sql, (config_name,)).fetchall())


def extract_context_texts(ctx) -> list[str]:
    if not ctx:
        return ['']
    if isinstance(ctx, str):
        try: ctx = json.loads(ctx)
        except Exception: return ['']
    if not isinstance(ctx, list):
        return ['']
    texts = []
    for item in ctx:
        text = item.get('text', '') or item.get('content', '')
        source = item.get('source', '')
        title = item.get('title', '')
        if text:
            header = f'[{source}] {title}' if source else title
            texts.append(f'{header}\n{text}' if header else text)
    return texts if texts else ['']


def save_metrics(run_id: int, new_metrics: dict):
    global _db_conn
    conn = _get_conn()
    try:
        conn.execute(
            "UPDATE goldset_runs SET metrics = COALESCE(metrics, '{}'::jsonb) || %s::jsonb WHERE id = %s",
            (json.dumps(new_metrics), run_id),
        )
        conn.commit()
    except Exception:
        try: conn.rollback()
        except Exception: pass
        _db_conn = None
        conn = _get_conn()
        conn.execute(
            "UPDATE goldset_runs SET metrics = COALESCE(metrics, '{}'::jsonb) || %s::jsonb WHERE id = %s",
            (json.dumps(new_metrics), run_id),
        )
        conn.commit()


for cfg in CONFIGS_TO_PROCESS:
    df = load_runs(cfg)
    print(f'  {cfg}: {len(df)} runs with gold_answer to evaluate')

In [ ]:
def run_batched(df: pd.DataFrame, save_to_db: bool = True) -> list:
    n_batches = (len(df) + BATCH_SIZE - 1) // BATCH_SIZE
    all_results = []
    t_start = time.time()

    for b in range(n_batches):
        start = b * BATCH_SIZE
        end = min(start + BATCH_SIZE, len(df))
        batch = df.iloc[start:end]
        print(f'  Batch {b+1}/{n_batches} [{start+1}-{end}/{len(df)}]', end=' ')

        try:
            ds = Dataset.from_dict({
                'question': batch['question'].tolist(),
                'answer': batch['response'].tolist(),
                'contexts': [extract_context_texts(row['retrieved_context']) for _, row in batch.iterrows()],
                'ground_truth': [str(row.get('gold_answer') or '') for _, row in batch.iterrows()],
            })

            tb = time.time()
            result = ragas_evaluate(ds, metrics=METRICS, llm=ragas_llm, embeddings=ragas_embeddings)
            scores = result.to_pandas()
            elapsed = time.time() - tb

            if save_to_db:
                sfx = SUFFIX
                for i, (_, row) in enumerate(batch.iterrows()):
                    if i >= len(scores): break
                    m = {f'judge_model{sfx}': JUDGE_MODEL, f'computed_at{sfx}': datetime.now().isoformat(), 'method': 'ragas'}
                    for key in METRIC_KEYS:
                        if key in scores.columns:
                            val = scores.iloc[i][key]
                            if pd.notna(val):
                                m[f'{key}{sfx}'] = round(float(val), 4)
                    save_metrics(row['run_id'], m)

            avgs = {k: scores[k].dropna().mean() for k in METRIC_KEYS if k in scores.columns and scores[k].dropna().shape[0] > 0}
            summary = ', '.join(f'{k}={v:.3f}' for k, v in avgs.items())
            print(f'{elapsed:.0f}s — {summary}')
            all_results.append(scores)

        except Exception as e:
            print(f'FAILED: {str(e)[:120]}')

    elapsed_total = time.time() - t_start
    print(f'  Done in {elapsed_total/60:.1f} min')
    return all_results

print('run_batched ready')

## Test sur 10 questions

In [ ]:
cfg = CONFIGS_TO_PROCESS[0]
df_test = load_runs(cfg).head(10)
print(f'Test: {cfg} — {len(df_test)} runs with gold_answer')

if len(df_test) > 0:
    test_results = run_batched(df_test, save_to_db=True)
    if test_results:
        scores = pd.concat(test_results)
        for k in METRIC_KEYS:
            if k in scores.columns:
                vals = scores[k].dropna()
                if len(vals) > 0:
                    print(f'  {k}: mean={vals.mean():.3f} | min={vals.min():.3f} | max={vals.max():.3f}')
else:
    print('Aucune question avec gold_answer — ce notebook ne peut tourner que sur des questions ayant une référence.')

## Full run

In [ ]:
for cfg in CONFIGS_TO_PROCESS:
    df = load_runs(cfg)
    if len(df) == 0:
        print(f'{cfg}: all done (or no gold_answer questions)')
        continue
    print(f'\n{"="*60}')
    print(f'{cfg}: {len(df)} runs — {JUDGE_MODEL}')
    run_batched(df, save_to_db=True)

## Vérification

In [ ]:
for cfg in CONFIGS_TO_PROCESS:
    remaining = load_runs(cfg)
    with psycopg.connect(DSN, row_factory=dict_row) as conn:
        total = conn.execute(
            "SELECT COUNT(*) as cnt FROM goldset_runs WHERE config_name = %s AND response IS NOT NULL",
            (cfg,)
        ).fetchone()['cnt']
        gold_total = conn.execute(
            """SELECT COUNT(*) as cnt FROM goldset_runs gr
               JOIN goldset_questions_v2 gq ON gr.question_id = gq.id
               WHERE gr.config_name = %s AND gr.response IS NOT NULL
               AND gq.gold_answer IS NOT NULL AND gq.gold_answer != ''""",
            (cfg,)
        ).fetchone()['cnt']
    done = gold_total - len(remaining)
    print(f'{cfg}: {done}/{gold_total} gold runs done, {len(remaining)} remaining ({total} total runs)')